# Code-MARL Quickstart
**Multi-Agent RL for Code Generation with GRPO**

This notebook walks through the full system end-to-end:
1. Test the sandbox (code executor)
2. Run a single debate episode (Coder ↔ Critic)
3. Launch GRPO training on HumanEval
4. View metrics in W&B

**Recommended runtime:** Google Colab A100 (free tier works for the 1.5B model)

## Step 0 — Install dependencies

In [ ]:
!pip install -q torch transformers trl datasets accelerate peft anthropic wandb python-dotenv

In [ ]:
# Mount Google Drive if using Colab
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print('Not in Colab — running locally')

In [ ]:
import os
import sys

# Set your API keys here (or use a .env file)
os.environ['ANTHROPIC_API_KEY'] = 'your-key-here'   # for debate loop
os.environ['WANDB_API_KEY']     = 'your-key-here'   # for experiment tracking
os.environ['HF_TOKEN']          = 'your-key-here'   # for Qwen model download

# Add project root to Python path
PROJECT_ROOT = '/content/code-marl'  # change if running locally
sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

## Step 1 — Test the sandbox

The sandbox is the most critical piece. If it doesn't work, nothing else will.

In [ ]:
from environment.sandbox import execute_code

# Correct solution → should get reward 1.0
r1 = execute_code(
    code='def add(a, b): return a + b',
    test_code='assert add(2, 3) == 5\nassert add(-1, 1) == 0'
)
print('Correct solution:', r1)
assert r1['reward'] == 1.0

# Wrong solution → should get reward 0.0
r2 = execute_code(
    code='def add(a, b): return 0',
    test_code='assert add(2, 3) == 5'
)
print('Wrong solution:', r2)
assert r2['reward'] == 0.0

print('\nSandbox tests passed!')

## Step 2 — Run a single debate episode

In [ ]:
from agents.debate import run_debate

problem = '''
def fizzbuzz(n: int) -> list[str]:
    """Return list of strings 1..n.
    Multiples of 3 → 'Fizz', multiples of 5 → 'Buzz',
    multiples of both → 'FizzBuzz', else the number as string."""
'''

tests = '''
result = fizzbuzz(15)
assert result[0] == '1'
assert result[2] == 'Fizz'
assert result[4] == 'Buzz'
assert result[14] == 'FizzBuzz'
assert len(result) == 15
'''

outcome = run_debate(problem=problem, test_code=tests, n_rounds=2, verbose=True)

print(f"\nReward before debate: {outcome['reward_initial']}")
print(f"Reward after debate:  {outcome['reward_final']}")
print(f"Improvement:          {outcome['debate_improvement']:+.4f}")

## Step 3 — Load a few HumanEval problems

In [ ]:
from environment.dataset import load_humaneval, format_problem_for_coder

problems = load_humaneval(max_problems=5)
for p in problems:
    print(f"{p['problem_id']}: {p['entry_point']}")
    print(f"  Prompt (first 100 chars): {p['prompt'][:100].strip()}...")
    print()

## Step 4 — Run GRPO training

This will:
- Load Qwen2.5-Coder-1.5B-Instruct
- Sample 4 completions per problem
- Compute rewards via sandbox
- Update model with GRPO
- Log reward curves to W&B

⚠️ **Requires a GPU (A100 recommended).** Estimated time: ~30 min for 50 problems.

In [ ]:
import wandb
wandb.init(project='code-marl', name='grpo-qwen-quickstart')

from training.grpo_trainer import train

train(
    max_problems=20,      # start small, increase later
    num_train_epochs=1,
    num_generations=4,
    output_dir='./checkpoints/quickstart',
)

## Step 5 — Evaluate pass@k

In [ ]:
from evaluation.metrics import evaluate_pass_at_k
from agents.coder import CoderAgent

agent = CoderAgent()
eval_problems = load_humaneval(max_problems=10)

# Generate 5 completions per problem
all_completions = []
for p in eval_problems:
    prompt = format_problem_for_coder(p)
    completions = [agent.write_solution(prompt)['raw'] for _ in range(5)]
    all_completions.append(completions)

metrics = evaluate_pass_at_k(eval_problems, all_completions, k_values=[1, 5])
print('Evaluation metrics:', metrics)